In [10]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split


In [11]:
full_df = pd.read_csv('../data/filtered/hotelrec_filtered_00001.csv')
ratings_df = full_df.sample(n=1_000_000, random_state=42)

print(f"\nOriginal number of ratings: {len(ratings_df)}")
ratings_df.drop_duplicates(subset=['user_id', 'item_id'], keep='last', inplace=True)
print(f"Number of unique ratings after de-duplication: {len(ratings_df)}")


Original number of ratings: 1000000
Number of unique ratings after de-duplication: 969053


In [12]:
# --- 2. Feature Preparation ---
print("\nPreparing features...")
feature_cols = [col for col in ratings_df.columns if col not in ['user_id', 'item_id', 'rating']]
item_features_df = ratings_df[['item_id'] + feature_cols].copy()
item_features_df_long = pd.melt(item_features_df, id_vars='item_id', value_vars=feature_cols, var_name='feature_name', value_name='feature_value')
item_features_df_long.dropna(inplace=True)
item_features_df_long['feature'] = item_features_df_long['feature_name'] + ":" + item_features_df_long['feature_value'].astype(str)
all_item_features = item_features_df_long['feature'].unique()
item_feature_list = item_features_df_long.groupby('item_id')['feature'].apply(list)


Preparing features...


In [13]:
# --- 3. Build Dataset and Split DataFrames FIRST ---
# To properly handle weights, we split the DataFrame before building matrices.
print("Splitting DataFrame into training and testing sets...")
train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42, shuffle=True)

# Build the dataset from the full data to ensure all users/items/features are known
dataset = Dataset()
dataset.fit(users=ratings_df['user_id'], items=ratings_df['item_id'], item_features=all_item_features)
def feature_list_generator(mapping):
    for item_id, features in mapping.items():
        yield (item_id, features)
item_features_matrix = dataset.build_item_features(feature_list_generator(item_feature_list))

# Build separate train and test matrices
print("Building train and test interaction matrices...")
(train_interactions, train_weights) = dataset.build_interactions((row.user_id, row.item_id, row.rating) for index, row in train_df.iterrows())
(test_interactions, test_weights) = dataset.build_interactions((row.user_id, row.item_id, row.rating) for index, row in test_df.iterrows())

Splitting DataFrame into training and testing sets...
Building train and test interaction matrices...


In [14]:
# --- 4. Model Training with Weights ---
# We still use 'warp' loss, but the weights will guide it to focus on high-rated items.
# Other good options are 'bpr' or 'logistic'. 'warp' is often a strong default.
model = LightFM(loss='warp',
                random_state=42,
                learning_rate=0.001,
                no_components=50,  # Increased components
                item_alpha=1e-4,   # Added regularization
                user_alpha=1e-4)   # Added regularization

print("\nTraining the LightFM model with ratings as weights...")
model.fit(train_interactions,
          sample_weight=train_weights,  # **THE CRITICAL FIX IS HERE**
          item_features=item_features_matrix,
          epochs=100, # Increased epochs
          num_threads=4,
          verbose=True)
print("Training complete.")


Training the LightFM model with ratings as weights...


Epoch:  43%|████▎     | 43/100 [02:06<02:44,  2.88s/it]

In [ ]:
# --- 5. Comprehensive Evaluation ---
print("\n--- Model Evaluation ---")
# The AUC will now be much more meaningful.
# train_auc = auc_score(model, train_interactions, item_features=item_features_matrix, num_threads=12).mean()
test_auc = auc_score(model, test_interactions, train_interactions=train_interactions, item_features=item_features_matrix, num_threads=12).mean()

# print(f"\nAUC Score on Training Set: {train_auc:.4f}")
print(f"AUC Score on Test Set:     {test_auc:.4f}")


--- Model Evaluation ---

AUC Score on Training Set: 0.8175
AUC Score on Test Set:     0.5527


In [ ]:
# --- 6. (Optional but Recommended) Hyperparameter Tuning with Optuna ---
# To run this, you would need to `pip install optuna`
# This code provides a template for finding the best parameters.

import optuna

def objective(trial):
    # Define the search space for hyperparameters
    no_components = trial.suggest_int('no_components', 20, 100)
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True)
    item_alpha = trial.suggest_float('item_alpha', 1e-6, 1e-2, log=True)
    user_alpha = trial.suggest_float('user_alpha', 1e-6, 1e-2, log=True)

    model = LightFM(loss='warp',
                    random_state=42,
                    no_components=no_components,
                    learning_rate=learning_rate,
                    item_alpha=item_alpha,
                    user_alpha=user_alpha)

    model.fit(train_interactions,
              sample_weight=train_weights,
              item_features=item_features_matrix,
              epochs=10, # Use fewer epochs during tuning for speed
              num_threads=4,
              verbose=False)

    # Evaluate on the test set
    test_auc = auc_score(model, test_interactions, train_interactions=train_interactions, item_features=item_features_matrix, num_threads=10).mean()

    return test_auc

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # Run 50 trials

print("\nBest trial:")
trial = study.best_trial
print(f"  AUC: {trial.value:.4f}")
print("  Best Parameters: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")